# QLoRA fine-tune: Qwen2.5-1.5B-Instruct on text-to-SQL

Made for a free Colab T4 (16GB). Base model loads in 4-bit NF4, LoRA adapters train on top. One epoch over 8k examples takes roughly 2-3 hours.

Runtime > Change runtime type > T4 GPU before running anything.

In [1]:
# colab already ships torch/transformers/peft/accelerate. only add what's missing,
# and upgrade bitsandbytes (the preinstalled/older one breaks against colab's new triton)
!pip install -q -U bitsandbytes trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00


In [2]:

import torch
print(torch.cuda.get_device_name(0))

# adapter gets saved to Drive at the end so it survives the runtime
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/qwen25-sql-lora"

Tesla T4
Mounted at /content/drive


In [3]:
from datasets import load_dataset

# processed split straight from the repo, no clone needed
DATA_URL = "https://raw.githubusercontent.com/Akshu24Tech/text2sql-qlora/main/data/train.jsonl"
train_ds = load_dataset("json", data_files=DATA_URL, split="train")
print(train_ds)
train_ds[0]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['question', 'context', 'answer'],
    num_rows: 8000
})


{'question': 'Which event in 2007 had a position of 5th?',
 'context': 'CREATE TABLE table_name_87 (event VARCHAR, year VARCHAR, position VARCHAR)',
 'answer': 'SELECT event FROM table_name_87 WHERE year = 2007 AND position = "5th"'}

## Prompt format

Each example becomes a short chat: system prompt fixes the task, user turn carries the schema + question, assistant turn is the target SQL. Using the model's own chat template so inference later looks exactly like training.

In [4]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM = "You are a text-to-SQL assistant. Given a table schema and a question, reply with only the SQL query, nothing else."

def to_chat_text(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
        {"role": "assistant", "content": row["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = train_ds.map(to_chat_text, remove_columns=train_ds.column_names)
print(train_ds[0]["text"])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

<|im_start|>system
You are a text-to-SQL assistant. Given a table schema and a question, reply with only the SQL query, nothing else.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE table_name_87 (event VARCHAR, year VARCHAR, position VARCHAR)

Question: Which event in 2007 had a position of 5th?<|im_end|>
<|im_start|>assistant
SELECT event FROM table_name_87 WHERE year = 2007 AND position = "5th"<|im_end|>



## Model in 4-bit + LoRA config

NF4 quantization with fp16 compute (T4 has no bf16). LoRA r=16 on all attention and MLP projections.

In [5]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # no kv cache during training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen25-sql-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    dataset_text_field="text",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=25,
    save_strategy="no",  # saving manually at the end
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# newer trl/peft create the LoRA weights in bf16, which crashes the fp16 grad
# scaler on the first optimizer step (T4 has no bf16). upcast trainables to fp32,
# the standard QLoRA setup: fp32 adapters, fp16 compute for the frozen base.
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.to(torch.float32)

trainer.model.print_trainable_parameters()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
25,1.330204
50,0.653526
75,0.631851
100,0.635536
125,0.612561
150,0.605202
175,0.600469
200,0.580184


In [ ]:
# save just the LoRA adapter (~70MB) to Drive, the eval notebook picks it up from there
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!ls -lh {SAVE_DIR}

In [ ]:
# sanity check: make sure the saved adapter actually contains trained weights.
# a fresh LoRA init has all lora_B matrices at exactly zero -- if that's what got
# saved, training ran in a different (dead) session and the weights are lost.
from safetensors import safe_open

with safe_open(f"{SAVE_DIR}/adapter_model.safetensors", framework="pt") as f:
    b_max = max(f.get_tensor(k).abs().max().item() for k in f.keys() if "lora_B" in k)
assert b_max > 0, "saved adapter is UNTRAINED (all lora_B are zero) -- re-run trainer.train() and save again in the same session"
print(f"adapter looks trained, max |lora_B| = {b_max:.4f}")

In [ ]:
# quick vibe check before proper eval
sample = {
    "context": "CREATE TABLE employees (name VARCHAR, department VARCHAR, salary INTEGER)",
    "question": "What is the average salary in the engineering department?",
}
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"Schema:\n{sample['context']}\n\nQuestion: {sample['question']}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = trainer.model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))